In [1]:
#| default_exp lawa

In [2]:
#| hide
import nbdev; nbdev.nbdev_export()

/usr/local/lib/python3.12/dist-packages/nbdev/export.py:80: UserWarning: Notebook '/workspaces/gpt/rugptxl_converter.ipynb' uses `#|export` without `#|default_exp` cell.
Note nbdev2 no longer supports nbdev1 syntax. Run `nbdev_migrate` to upgrade.
See https://nbdev.fast.ai/getting_started.html for more information.
  warn(f"Notebook '{nbname}' uses `#|export` without `#|default_exp` cell.\n"


In [3]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "3"
os.environ["VLLM_ALLOW_DEPRECATED_BEAM_SEARCH"] = "1"
#os.environ['VLLM_LOGGING_LEVEL'] = 'DEBUG'

In [4]:
#| export
from os import getenv
from vllm import LLM, SamplingParams
from transformers import AutoTokenizer
from front.common import process_seq
import random
model_path = getenv("MODEL")
gpu_part = int(getenv("gpu_fraction",40))/100

In [ ]:
gpu_part=0.2

In [6]:
# Step 0: loss: 3.4908 lr (3e-5)/2 final Loss: 3.1622
model_path = 'large/poetry'


# loss 1.43 for llama 3.2 1B

# Step 0: 4.2097 lr (3e-5)/2 final Loss: 2.6410
model_path = 'xl/pelevin'

# Step 0: 5.5830 lr (3e-5)/2 final Loss: 2.6281
model_path = 'mig'

# 3e-5 Train 3.193800 valid Loss: 2.987608
model_path = 'xl/poetry'

# Step 0: 2.7811 lr 3e-6 final Loss: 2.5810
model_path = 'lawa'

# Step 0: loss: 2.9164 lr (3e-5)/2 final Loss: 2.7707
model_path = 'large/pelevin'


In [7]:
#| export
full_path = f'./models/{model_path}'
tokenizer = AutoTokenizer.from_pretrained(full_path)
model = LLM(model=full_path, dtype="bfloat16", device="cuda", gpu_memory_utilization=gpu_part)

INFO 10-06 13:02:17 config.py:1652] Downcasting torch.float32 to torch.bfloat16.
INFO 10-06 13:02:17 llm_engine.py:226] Initializing an LLM engine (v0.6.1.dev238+ge2c6e0a82) with config: model='./models/large/pelevin', speculative_config=None, tokenizer='./models/large/pelevin', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, override_neuron_config=None, rope_scaling=None, rope_theta=None, tokenizer_revision=None, trust_remote_code=False, dtype=torch.bfloat16, max_seq_len=1024, download_dir=None, load_format=LoadFormat.AUTO, tensor_parallel_size=1, pipeline_parallel_size=1, disable_custom_all_reduce=False, quantization=None, enforce_eager=False, kv_cache_dtype=auto, quantization_param_path=None, device_config=cuda, decoding_config=DecodingConfig(guided_decoding_backend='outlines'), observability_config=ObservabilityConfig(otlp_traces_endpoint=None, collect_model_forward_time=False, collect_model_execute_time=False), seed=0, served_model_name=./models/large/pelevin, use_v

[W1006 13:02:18.504436396 socket.cpp:697] [c10d] The client socket cannot be initialized to connect to [bbb]:37371 (errno: 97 - Address family not supported by protocol).


Loading safetensors checkpoint shards:   0% Completed | 0/1 [00:00<?, ?it/s]


INFO 10-06 13:02:20 model_runner.py:1025] Loading model weights took 1.4419 GB
INFO 10-06 13:02:20 gpu_executor.py:122] # GPU blocks: 908, # CPU blocks: 1456
INFO 10-06 13:02:21 model_runner.py:1329] Capturing the model for CUDA graphs. This may lead to unexpected consequences if the model is not static. To run the model in eager mode, set 'enforce_eager=True' or use '--enforce-eager' in the CLI.
INFO 10-06 13:02:21 model_runner.py:1333] CUDA graphs can take additional 1~3 GiB memory per GPU. If you are running out of memory, consider decreasing `gpu_memory_utilization` or enforcing eager mode. You can also reduce the `max_num_seqs` as needed to decrease memory usage.
INFO 10-06 13:02:36 model_runner.py:1456] Graph capturing finished in 15 secs.


In [7]:
model.llm_engine.model_config.max_model_len

2048

In [8]:
#| export
def iftoken(tokenizer, tokens):
    # returns token id if the given string is one token
    token_ids = [tokenizer.encode(token, add_special_tokens=False) for token in tokens]
    return [id for sublist in token_ids for id in sublist if len(sublist) == 1]

stop_token_ids = iftoken(tokenizer, ['<|endoftext|>','|eot_id|','<|end_of_text|>','<|eot_id|>','<pad>'])

def create_token_blocker(tokenizer, blocked_tokens):
    blocked_token_ids = iftoken(tokenizer, blocked_tokens)
    
    def token_blocker(input_ids, scores):
        if scores.dim() == 2:
            scores[:, list(blocked_token_ids)] = -float('inf')
        elif scores.dim() == 1:
            scores[list(blocked_token_ids)] = -float('inf')
        else:
            raise ValueError(f"Unexpected score tensor shape: {scores.shape}")
        return scores
    
    return token_blocker
    
def get_sampling_params(tokenizer, length: int, num_samples: int, allow_linebreak: bool, temperature: float):
    blocked_tokens = ['?»',"».",'.\n','\n\t\t','http://',',[','("','.]',' («',')','\u2004',']','(«','[', ' [', '(', ' (', '\xa0', '*', '­', '~', '_', '\\', '\uf04a', '\ufeff', '\u2028']
    if not allow_linebreak:
        blocked_tokens.extend(['\n', '\n\n',' \n'])
    
    token_blocker = create_token_blocker(tokenizer, blocked_tokens)
    return SamplingParams(
        max_tokens=length,
        n=num_samples,
        top_k=-1,
        stop_token_ids=stop_token_ids,
        ignore_eos=True,
        logits_processors=[token_blocker],
        repetition_penalty=2.,
        temperature=temperature,
        top_p=0.9,
        seed=random.randint(0, 1000000),
#        use_beam_search=True,
#        best_of=num_samples*4,
#        temperature=0.,
#        top_p=1.,
         #penalty_alpha=0.6, top_k=4
    )

In [9]:
stop_token_ids

[5, 1]

In [13]:
#| export
def get_sample(prompt: str, length: int, num_samples: int, allow_linebreak: bool, temperature: float = 1.0):
    max_input = model.llm_engine.model_config.max_model_len - length
    prompt = tokenizer.decode(tokenizer.encode(prompt)[-max_input:]).removeprefix('<|begin_of_text|>')
    sampling_params = get_sampling_params(tokenizer, length, num_samples, allow_linebreak, temperature)
    outputs = model.generate(prompt, sampling_params)
    generated_sequences = [oo.text for o in outputs for oo in o.outputs]
    return process_seq(generated_sequences)

In [11]:
%%time
get_sample('На словах ты Лев Толстой, а на деле'*100000, 300, 4, False)

Token indices sequence length is longer than the specified maximum sequence length for this model (1200000 > 2048). Running this sequence through the model will result in indexing errors
Processed prompts: 100%|██████████| 1/1 [00:02<00:00,  2.56s/it, est. speed input: 683.92 toks/s, output: 469.51 toks/s]

CPU times: user 6.64 s, sys: 919 ms, total: 7.56 s
Wall time: 7.53 s


['. Нашь и чита отвеша такая! Астрый… Има о томаю-тина твоем». Вох-зайфак накладухин пошел бывает мудя тебестауверху вашкада сибирский хуй померобарана? – бараньяноват дума нет правят дохнули стишкиныча боится будешиной мортишу базарва шум неясно написалище кончкохасианином заката бляповоллупалек лам минусом повернулитянчик из «Абулах ебоконШейджрапедаев», чтили гром улыкнул прочит аллафутил весь деньгаеткин“, я просила жестимой Аббасыльбажурман зовские слова „Вот во время такое слово для того места натвак полдзе ханку читать надо“ Дурам соратых мужчина пидеевичи белого днями джих арабских паццагдырнисьминутьсянью“.',
 '. Нашь и хуй помолтва шу-на слова «В». ИакинО! Астрый… Вотан мох-то время отстаютарилупаулина таксиджал бываетя тебегают – бараномон барахнуло за столыплем тврубачевыча оралмаша нетрвуратоват духаем поеткинью чита вродины сели изволли белых робе мордарапедаев до конца этажим неясно пьямазо боишки мацвой гейбирских паренькафардыкину хотятоколаек жопаешиной минус заката к

In [14]:
%%time
get_sample('На словах ты Лев Толстой, а на деле', 300, 4, False)

Processed prompts: 100%|██████████| 1/1 [00:01<00:00,  1.73s/it, est. speed input: 6.95 toks/s, output: 695.49 toks/s]

CPU times: user 1.73 s, sys: 3.36 ms, total: 1.74 s
Wall time: 1.73 s


[' – весь этот белый халатик с золотыми пуговицами. А если я правильно понял? Вот оно и есть твое настоящее имя! Или это тоже часть фамилии…» Я не мог отделаться от чувства юмора: в его голосе звучало как бы что-то похожее надменное; он был уверен даже во сне до того момента когда меня вырвало из него навсегда за день или два назад по той же самой причине). Но мне было страшновато смотреть именно так — под черным небом мерцали звезды со всех сторон сразу после моих усилий дать им команду самих себя выбрать место для встречи между ними лично у них внутри темной комнаты вокруг шкафа «Житан», где они встречались несколько часов подряд без всякой помощи человеческого труда обретая спокойствие перед лицемерными зеркалами своего внутреннего мира совершенно незаметных глазу глазниц сквозь толщу земли коридоров двухэтажного пансионата рядом бетонный ларьк давности Кремля».',
 ' – я. Ты мне сейчас скажи: в чем разница? В том и заключается самое главное…» Я не понял сначала смысла этих выражений